# Annexe B — Cahier de code, Chapitre 9
## Le flot optique

Ce notebook accompagne le chapitre 9 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Deux images consécutives : la seconde est la première décalée
import numpy as np
from skimage import data, color
from scipy.ndimage import shift

f1 = color.rgb2gray(data.astronaut())
f2 = shift(f1, shift=(0, 3))     # décalage de 3 px vers la droite

## 9.1 — La contrainte du flot optique

In [ ]:
# Ix·u + Iy·v + It = 0   (une équation, deux inconnues)
Iy, Ix = np.gradient(f1)
It = f2 - f1

## 9.2 — Le problème d'ouverture

In [ ]:
# seule la composante du flot normale au bord est observable
norme = np.hypot(Ix, Iy) + 1e-9
flot_normal = -It / norme        # vitesse le long du gradient

## 9.3 — Lucas-Kanade (épars)

In [ ]:
# suit quelques points saillants entre f1 et f2 (OpenCV, uint8)
import cv2
a = (f1 * 255).astype(np.uint8); b = (f2 * 255).astype(np.uint8)
p0 = cv2.goodFeaturesToTrack(a, maxCorners=100, qualityLevel=0.3, minDistance=7)
p1, st, err = cv2.calcOpticalFlowPyrLK(a, b, p0, None)

## 9.4 — Horn-Schunck (global / dense)

In [ ]:
# flot dense avec hypothèse de régularité (variante TV-L1 de skimage)
from skimage.registration import optical_flow_tvl1
v, u = optical_flow_tvl1(f1, f2)   # composantes verticale et horizontale

## 9.5 — Flot épars ou dense

In [ ]:
# dense : Farnebäck calcule un vecteur par pixel
import cv2
a = (f1 * 255).astype(np.uint8); b = (f2 * 255).astype(np.uint8)
flot = cv2.calcOpticalFlowFarneback(a, b, None,
        0.5, 3, 15, 3, 5, 1.2, 0)   # → H×W×2